In [1]:
import sys
# sys.path.append(r'C:\Users\Lehnert Lab\Documents\GitHub\measurement\pna_control')
sys.path.append('..')
import pna_control as PNA
import numpy as np
import os

# for method in dir(PNA):""
#     print(method)


%load_ext autoreload
%autoreload 2
%aimport

Modules to reload:
all-except-skipped

Modules to skip:



In [2]:
# from gpib_ctypes import make_default_gpib
# make_default_gpib()

# import pygpib as gpib
# print(gpib.list_adapters())

import pyvisa
# rm = pyvisa.ResourceManager('@py')
rm     =     pyvisa.ResourceManager()
display(rm.list_resources())

('TCPIP0::K-N5231B-57006::hpib7,1::INSTR',
 'TCPIP0::K-N5231B-57006::hpib7,2::INSTR',
 'TCPIP0::K-N5231B-57006::hpib7,3::INSTR',
 'TCPIP0::K-N5231B-57006::hpib7,4::INSTR',
 'TCPIP0::K-N5231B-57006::hpib7,5::INSTR',
 'TCPIP0::K-N5231B-57006::hpib7,6::INSTR',
 'TCPIP0::K-N5231B-57006::hpib7,7::INSTR',
 'TCPIP0::K-N5231B-57006::hpib7::INSTR',
 'TCPIP0::K-N5231B-57006::inst0::INSTR',
 'TCPIP0::169.254.89.124::INSTR',
 'TCPIP0::192.168.137.10::PNA::INSTR',
 'TCPIP0::K-N5231B-57006.local::hislip0::INSTR',
 'TCPIP0::K-N5231B-57006.local::inst0::INSTR',
 'ASRL1::INSTR',
 'ASRL3::INSTR',
 'GPIB0::7::INSTR',
 'TCPIP0::169.254.89.124::hislip0::INSTR',
 'TCPIP0::169.254.89.124::inst0::INSTR',
 'TCPIP0::192.168.137.10::hislip0::INSTR')

In [3]:
#set up the pna to measure s21 for the specific instrument gpib0::16::instr
sys.path.append('../instrument_control/')
sys.path.append('../temperature_control/')
sys.path.append('../pna_control/')

from anritsu import AnritsuCtrl

gpib_addr = 'GPIB0::7::INSTR'
instr_addr = 'TCPIP0::169.254.89.124::hislip0::INSTR'

keysight = rm.open_resource(instr_addr)
anritsu = rm.open_resource(gpib_addr)


In [4]:
temperature = 12e-3
sampleid = 'test_02' #project id followed by sample number and die number
edelay = 73.05 #ns

print(os.getcwd())

e:\GitHub\bcqt-ctrl\test


## Broadband measurement

In [15]:
# start with a broadband sweep
sys.path.append('./broadband/')
from user_ctrl_broadband import measure_multiple_resonators

# Set the center frequencies, spans, delays, powers
fcs = [4.5, 5.5, 6.5, 7.5]
spans = [1000]*len(fcs)
delays = [62]*len(fcs)
powers = np.linspace(-30, -35, 2)

# Change the sample name

# measure_multiple_resonators(fcs, spans, delays, powers,
#         ifbw=1000., sparam='S21', npts=65,
#         adaptive_averaging=False, sample_name=sampleid,
#         runtime=0., cal_set = None)


#### Note about user_fit and user_fit_broadband

these use the old version of scresonators, before scott's refactoring
if you're seeing errors, the first thing you need to check is the version of scresonators
use `git checkout 8bad279` to switch to the pre-factorization commit, and something like `git checkout HEAD` to switch back

In [58]:
## stripped out this portion from user_fit.py from the broadband folder
from user_fit_broadband import stitch_broadband

# set input temperature
temperature = 20e-3

# Set the input powers and temperature string
powers_hi = np.linspace(-15, -35, 5)
powers_lo =  np.linspace(-40, -95, 12)
powers_in = np.hstack((powers_hi, []))
                        # [-97, -99, -100, -103, -105, -107]))
                        
# Crop out high power points with strong quasiparticle response
# powers_in = powers_in[6:]
# print(powers_in)

tstr = f'{temperature*1e3:.0f}mK'

fc = fcs[0]
fstr = f'{np.floor(fc):.1f}_{fc:.1f}GHz'.replace('.', 'p')

# Set the filename strings
filenames_in = [f'{sampleid}_{dstr}_{fstr}_{int(p)}dB_{tstr}.csv' 
            for p in powers_in]
display(filenames_in)

# Concatenate broadband sweeps
freq_band = [4., 8.]
freq_step = 1.
dstr = '240328'
powers = [-30, -35]
stitch_broadband(prefix = sampleid, 
                 freq_band = freq_band, 
                 freq_step = freq_step, 
                 dstr = dstr, 
                 powers = powers,
                 Tmxc = 25, 
                 fscale = 1e9, 
                 sparam = 'S21')    
    


Loading user_fit_broadband from: e:\GitHub\bcqt-ctrl\test\broadband
                     parent dir: e:\GitHub\bcqt-ctrl\test


['test_02_240328_4p0_4p5GHz_-15dB_20mK.csv',
 'test_02_240328_4p0_4p5GHz_-20dB_20mK.csv',
 'test_02_240328_4p0_4p5GHz_-25dB_20mK.csv',
 'test_02_240328_4p0_4p5GHz_-30dB_20mK.csv',
 'test_02_240328_4p0_4p5GHz_-35dB_20mK.csv']

center_freqs: [4.5, 5.5, 6.5, 7.5]
center_freqs_strs: ['4p500', '5p500', '6p500', '7p500']
sdirs:  
['test_02_4p500GHz*', 'test_02_5p500GHz*', 'test_02_6p500GHz*', 'test_02_7p500GHz*']
glob:  ['240520_test_02\\test_02_4p500GHz_-30dB_9mK.csv', '240520_test_02\\test_02_4p500GHz_-35dB_9mK.csv', '240520_test_02\\test_02_5p500GHz_-30dB_9mK.csv', '240520_test_02\\test_02_5p500GHz_-35dB_9mK.csv', '240520_test_02\\test_02_6p500GHz_-30dB_9mK.csv', '240520_test_02\\test_02_6p500GHz_-35dB_9mK.csv', '240520_test_02\\test_02_7p500GHz_-30dB_9mK.csv', '240520_test_02\\test_02_7p500GHz_-35dB_9mK.csv']
dirs:  
[['240520_test_02\\test_02_4p500GHz_-30dB_9mK.csv', '240520_test_02\\test_02_4p500GHz_-35dB_9mK.csv', '240520_test_02\\test_02_5p500GHz_-30dB_9mK.csv', '240520_test_02\\test_02_5p500GHz_-35dB_9mK.csv', '240520_test_02\\test_02_6p500GHz_-30dB_9mK.csv', '240520_test_02\\test_02_6p500GHz_-35dB_9mK.csv', '240520_test_02\\test_02_7p500GHz_-30dB_9mK.csv', '240520_test_02\\test_02_7p500GHz_-35dB_9mK.csv

FileNotFoundError: ['240520_test_02\\test_02_4p500GHz_-30dB_9mK.csv', '240520_test_02\\test_02_4p500GHz_-35dB_9mK.csv', '240520_test_02\\test_02_5p500GHz_-30dB_9mK.csv', '240520_test_02\\test_02_5p500GHz_-35dB_9mK.csv', '240520_test_02\\test_02_6p500GHz_-30dB_9mK.csv', '240520_test_02\\test_02_6p500GHz_-35dB_9mK.csv', '240520_test_02\\test_02_7p500GHz_-30dB_9mK.csv', '240520_test_02\\test_02_7p500GHz_-35dB_9mK.csv']/test_02_4p500GHz_-30dB_25mK.csv not found.